# Multimodal RAG

> Notebook generated from [`examples/rag/09_multimodal_rag.py`](../../examples/rag/09_multimodal_rag.py).

| Data | Value |
|------|-------|
| **Dataset** | arXiv + MedQuAD + ATIS + ActivityNet (mix — local CSV + embedded) |
| **API key** | ⚠️  **Requires API key** (`ANTHROPIC_API_KEY` or `OPENAI_API_KEY`) in `.env`. |

**Expected result:** Indexes text/image/audio/video with `modality`+`source_uri`; demonstrates `search(query, modalities=[…])`.

---

## Setup

```bash
uv pip install -e ".[dev,all]"
```

> To use `await main()` directly in cells, this notebook
> installs `nest_asyncio` in the first cell.

---

## Original docstring

```
Multimodal RAG — Index and search text + images + audio in a single collection
=============================================================================
Architecture: SPEC-MM-RAG-001 / prismal.rag.multimodal

Dataset: arXiv (cs.CV/cs.CL) + MedQuAD + ATIS (synthetic multimodal mix)
  • Texts    : arXiv abstracts (`Langgraph_tutorials/data/arxiv/...`)
               + MedQuAD questions.
  • Images   : hypothetical figures associated with each paper (caption is
               the paper description) — PNG generated in memory.
  • Audio    : ATIS-style utterances with their transcript as caption.
  • Why: the `MultimodalRAGEngine` without a real cross-modal embedder
    indexes the *captions/transcripts* (DD-MM-003). This example shows
    how each modality is preserved in `metadata["modality"]` and how
    `MultimodalRAGEngine.search(modalities=[...])` filters results.

Architecture description:
  1. Load/generate documents per modality → all with
     `metadata["modality"]` ∈ {text, image, audio, video}.
  2. `engine.index(path)` or `store.add_documents(...)`: persist in
     ChromaDB (or in this demo, in an equivalent in-memory store).
  3. `engine.search(query, k=K, modalities=None)` → `[
     MultimodalRetrievedChunk(modality, content, score, source_uri, …)]`.
  4. Same `query`, different `modalities` → different results.
  5. Without a cross-modal embedder, the engine falls back to textual mode
     (`logger.warning("multimodal_rag_textual_fallback")`) — explicitly.

Uso:
    uv run python examples/rag/09_multimodal_rag.py
```

In [ ]:
# Enable nested event loop to use `await` directly in Jupyter.
import sys
if 'nest_asyncio' not in sys.modules:
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        %pip install -q nest_asyncio
        import nest_asyncio
        nest_asyncio.apply()

## Locate the dataset

This notebook reads data from `data/` at the repo root. The cell below locates that folder by walking up from the current cwd.

In [ ]:
# Resolve the repo's data/ folder by walking up until it is found.
# Replaces the DATA_ROOT
# the original script had when it lived inside prismal/examples/.
import pathlib


def _find_data_root() -> pathlib.Path:
    cur = pathlib.Path.cwd().resolve()
    for _ in range(8):
        if (cur / "data").is_dir():
            return cur / "data"
        if cur.parent == cur:
            break
        cur = cur.parent
    raise FileNotFoundError(
        "data/ folder not found. Run the notebook from the "
        "prisma-notebooks/ repo where data/ lives at the root."
    )


DATA_ROOT = _find_data_root()
print(f"DATA_ROOT = {DATA_ROOT}")

## Setup · imports and dataset

In [ ]:
from __future__ import annotations

import asyncio
import csv
import textwrap
from pathlib import Path

from langchain_core.documents import Document

from prismal.agents.multimodal import Modality
from prismal.rag.multimodal import MultimodalRAGEngine, MultimodalRetrievedChunk

ROOT = DATA_ROOT
ARXIV_CSV = ROOT / "arxiv" / "arxiv_papers.csv"
MEDQUAD_CSV = ROOT / "medquad" / "medquad.csv"

## In-memory ChromaVectorStore shim

In [ ]:
# `MultimodalRAGEngine` only needs:
#   - .add_documents(list[Document]) -> list[str]
#   - .similarity_search(query, k) -> list[(Document, score)]
# We implement a BM25-like substitute to avoid Chroma in this example.
class InMemoryStore:
    """Minimal mock of `ChromaVectorStore` using overlapping-keyword score."""

    def __init__(self) -> None:
        self._docs: list[Document] = []

    def add_documents(self, documents: list[Document]) -> list[str]:
        ids: list[str] = []
        for i, d in enumerate(documents):
            self._docs.append(d)
            ids.append(f"doc-{len(self._docs):04d}")
        return ids

    def similarity_search(self, query: str, k: int = 5) -> list[tuple[Document, float]]:
        q_terms = {t.lower() for t in query.split() if len(t) > 2}
        scored: list[tuple[Document, float]] = []
        for d in self._docs:
            doc_terms = {t.lower().strip(".,;:!?()[]") for t in d.page_content.split()}
            overlap = len(q_terms & doc_terms)
            denom = len(q_terms) or 1
            score = overlap / denom
            if score > 0:
                scored.append((d, float(score)))
        scored.sort(key=lambda x: -x[1])
        return scored[:k]

## Loading the multimodal corpus

In [ ]:
def _load_text_docs(limit: int = 8) -> list[Document]:
    """arXiv abstracts + MedQuAD questions/answers as TEXT modality."""
    out: list[Document] = []
    if ARXIV_CSV.exists():
        try:
            with ARXIV_CSV.open(encoding="utf-8") as fh:
                for i, row in enumerate(csv.DictReader(fh)):
                    if i >= limit // 2:
                        break
                    out.append(Document(
                        page_content=f"{row['title']}. {textwrap.shorten(row['abstract'], 300)}",
                        metadata={
                            "modality": "text",
                            "source_uri": f"arxiv://{row['arxiv_id']}",
                            "source": row["arxiv_id"],
                            "category": row.get("category", ""),
                        },
                    ))
        except Exception:  # noqa: BLE001
            pass
    if MEDQUAD_CSV.exists():
        try:
            with MEDQUAD_CSV.open(encoding="utf-8") as fh:
                for i, row in enumerate(csv.DictReader(fh)):
                    if i >= limit // 2:
                        break
                    out.append(Document(
                        page_content=f"Q: {row['question']} A: {textwrap.shorten(row['answer'], 280)}",
                        metadata={
                            "modality": "text",
                            "source_uri": f"medquad://{row['focus_area']}",
                            "source": row.get("focus_area", "medquad"),
                        },
                    ))
        except Exception:  # noqa: BLE001
            pass
    return out or _embedded_text_docs()


def _embedded_text_docs() -> list[Document]:
    return [
        Document(
            page_content="Retrieval-Augmented Generation combines retrieval with generation.",
            metadata={"modality": "text", "source_uri": "embedded://rag",
                      "source": "embedded-rag"},
        ),
        Document(
            page_content="Type-2 diabetes is a chronic metabolic disorder affecting glucose.",
            metadata={"modality": "text", "source_uri": "embedded://medquad/diabetes",
                      "source": "medquad/diabetes"},
        ),
    ]


def _image_docs() -> list[Document]:
    """Figure captions (no cross-modal embedding → textual fallback)."""
    return [
        Document(
            page_content="Figure: architecture diagram of a transformer encoder block "
                         "with multi-head attention and feed-forward layers.",
            metadata={"modality": "image", "source_uri": "arxiv://2604.02185#fig1",
                      "source": "arxiv-2604.02185", "type": "diagram"},
        ),
        Document(
            page_content="Figure: a histopathology slide showing glaucoma-induced optic "
                         "nerve atrophy in a 65-year-old patient.",
            metadata={"modality": "image", "source_uri": "medquad://glaucoma/fig",
                      "source": "medquad-glaucoma", "type": "medical-photo"},
        ),
        Document(
            page_content="Figure: bar chart comparing BM25, dense, and hybrid retrieval "
                         "MRR@10 across BEIR benchmark.",
            metadata={"modality": "image", "source_uri": "arxiv://2604.02077#fig3",
                      "source": "arxiv-2604.02077", "type": "chart"},
        ),
    ]


def _audio_docs() -> list[Document]:
    """Transcripts of ATIS-style utterances as AUDIO modality."""
    return [
        Document(
            page_content="Show me all flights from Boston to Denver on Tuesday morning.",
            metadata={"modality": "audio", "source_uri": "atis://flight_search_001",
                      "source": "atis-001", "language": "en"},
        ),
        Document(
            page_content="How many seats are available on the two p.m. flight.",
            metadata={"modality": "audio", "source_uri": "atis://seat_avail_003",
                      "source": "atis-003", "language": "es"},
        ),
        Document(
            page_content="Voice command transcript: cancel my reservation for tomorrow's flight.",
            metadata={"modality": "audio", "source_uri": "atis://cancellation_004",
                      "source": "atis-004", "language": "en"},
        ),
    ]


def _video_docs() -> list[Document]:
    return [
        Document(
            page_content="Video: cooking pasta from scratch — boiling water, adding spaghetti, "
                         "stirring with a wooden spoon, plating with sauce.",
            metadata={"modality": "video", "source_uri": "anet://cooking_pasta",
                      "source": "anet-001", "duration_s": 24.0},
        ),
        Document(
            page_content="Video: skateboarder lands a kickflip down a seven-stair set.",
            metadata={"modality": "video", "source_uri": "anet://skateboard_trick",
                      "source": "anet-002", "duration_s": 12.0},
        ),
    ]

## Main demo

In [ ]:
async def main() -> None:
    print("=" * 72)
    print("MultimodalRAGEngine · index + search text/image/audio/video")
    print("=" * 72)

    # 1. Build multimodal corpus
    store = InMemoryStore()
    text_docs = _load_text_docs(limit=6)
    image_docs = _image_docs()
    audio_docs = _audio_docs()
    video_docs = _video_docs()

    print(f"\nCorpus: {len(text_docs)} TEXT · {len(image_docs)} IMAGE "
          f"· {len(audio_docs)} AUDIO · {len(video_docs)} VIDEO")

    store.add_documents(text_docs + image_docs + audio_docs + video_docs)

    # 2. Instantiate engine (no cross_modal_embedder → textual fallback)
    engine = MultimodalRAGEngine(vector_store=store)  # type: ignore[arg-type]

    queries = [
        "transformer architecture and attention",
        "flight from Boston",
        "glaucoma",
        "skateboard tricks",
    ]

    # 3. Search without modality filter
    print("\n" + "─" * 72)
    print("1) search(query, k=3) — no modality filter")
    print("─" * 72)
    for q in queries:
        print(f"\n  query: {q!r}")
        chunks: list[MultimodalRetrievedChunk] = await engine.search(q, k=3)
        if not chunks:
            print("    (no results — keyword overlap=0)")
        for c in chunks:
            print(f"    [{c.modality.value:5}] score={c.score:.2f}  "
                  f"{c.source_uri}\n           → {c.content[:80]}…")

    # 4. Modality-filtered search
    print("\n" + "─" * 72)
    print("2) search(query, k=3, modalities=[IMAGE]) — images only")
    print("─" * 72)
    for q in ["transformer architecture", "glaucoma diagnosis"]:
        print(f"\n  query: {q!r}")
        chunks = await engine.search(q, k=3, modalities=[Modality.IMAGE])
        for c in chunks:
            assert c.modality is Modality.IMAGE
            print(f"    [IMAGE] {c.source_uri} score={c.score:.2f}")
            print(f"            → {c.content[:80]}…")
        if not chunks:
            print("    (no relevant images)")

    # 5. Multi-modality search (text + video)
    print("\n" + "─" * 72)
    print("3) search(query, k=4, modalities=[TEXT, VIDEO])")
    print("─" * 72)
    q = "cooking"
    chunks = await engine.search(q, k=4, modalities=[Modality.TEXT, Modality.VIDEO])
    print(f"\n  query: {q!r}")
    for c in chunks:
        assert c.modality in {Modality.TEXT, Modality.VIDEO}
        print(f"    [{c.modality.value:5}] {c.source_uri} score={c.score:.2f}")
        print(f"            → {c.content[:80]}…")

    print("\n" + "=" * 72)
    print("OK — modalities preserved in metadata, filtering works without a cross-modal embedder")
    print("=" * 72)


if __name__ == "__main__":
    asyncio.run(main())

---

## Run the demo

The original file ends with `asyncio.run(main())`. In Jupyter use:

```python
await main()
```

Thanks to the `nest_asyncio.apply()` in the first cell, the
`async` calls run without needing a separate event loop.

In [ ]:
# Uncomment to run the end-to-end demo (requires API key if applicable).
# await main()